In [ ]:
import pandas as pd
import numpy as np

# Load the 4 exported files
orders   = pd.read_csv('orders_clean_export.csv')
payments = pd.read_csv('payments_export.csv')
reviews  = pd.read_csv('reviews_export.csv')
items    = pd.read_csv('items_with_category_export.csv')

# Inspect each one — run this for all 4
for name, df in [('orders',orders),('payments',payments),
                 ('reviews',reviews),('items',items)]:
    print(f"\n{'='*40}")
    print(f"TABLE: {name}")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"\nColumn types:\n{df.dtypes}")
    print(f"\nMissing values:\n{df.isnull().sum()}")
    print(f"\nFirst 3 rows:\n{df.head(3)}")

Fix the column names before doing any operations.

In [12]:
# Step 1: Reload the CSV but tell pandas the file has NO header
# header=None means "don't treat row 1 as column names"
orders = pd.read_csv('orders_clean_export.csv', header=None)

# Step 2: Manually assign the correct column names
# These must match the exact columns from your SQL export query
orders.columns = [
    'order_id',
    'customer_id',
    'order_status',
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'customer_state',
    'customer_city'
]

# Step 3: Drop the first row — it contains the original "header"
# data that got pushed down into row 0
orders = orders.iloc[1:].reset_index(drop=True)

# Step 4: Verify it worked
print("Column names:")
print(orders.columns.tolist())

print("\nShape:", orders.shape)

print("\nFirst 2 rows:")
print(orders.head(2))



# payments_export.csv
payments = pd.read_csv('payments_export.csv', header=None)
payments.columns = ['order_id', 'payment_type',
                    'payment_installments', 'payment_value']
payments = payments.iloc[1:].reset_index(drop=True)

# reviews_export.csv
reviews = pd.read_csv('reviews_export.csv', header=None)
reviews.columns = ['order_id', 'review_score', 'review_comment_message']
reviews = reviews.iloc[1:].reset_index(drop=True)

# items_with_category_export.csv
items = pd.read_csv('items_with_category_export.csv', header=None)
items.columns = ['order_id', 'price', 'freight_value', 'category']
items = items.iloc[1:].reset_index(drop=True)

Column names:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_state', 'customer_city']

Shape: (99440, 9)

First 2 rows:
                           order_id                       customer_id  \
0  d2b0ee54c610c7e21c4ba15a4e5aafe2  c599a0db8bdda37b57b248a20a566446   
1  694e3d121f16bcbb9b4595b6c6ab3baf  2e0339ff984d6f41f11dfdbc7c93dcda   

  order_status     order_purchase_timestamp            order_approved_at  \
0    delivered  2018-01-17 15:17:29.0000000  2018-01-17 15:32:48.0000000   
1    delivered  2017-12-15 13:43:13.0000000  2017-12-16 16:20:28.0000000   

  order_delivered_customer_date order_estimated_delivery_date customer_state  \
0   2018-01-24 01:09:37.0000000   2018-02-07 00:00:00.0000000             SP   
1   2017-12-22 01:07:14.0000000   2018-01-10 00:00:00.0000000             BA   

  customer_city  
0     juquitiba  
1     ibotirama  



Fix the date columns — convert from text to datetime

Every date column loads as text — pandas cannot calculate with text dates

In [11]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

print(orders[date_columns].dtypes)
print("\nNulls after conversion:")
print(orders[date_columns].isnull().sum())

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Nulls after conversion:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64



Remove rows you cannot use for analysis

Decide which rows are genuinely useless — and only drop those specific rows




In [13]:
# DECISION 1: Keep only delivered orders for delivery analysis
# Orders in "shipped", "processing" etc. have no delivery date — useless for
# our late-delivery analysis but valid for revenue analysis
# So we create a separate delivered-only subset

delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f"All orders:       {len(orders):,}")
print(f"Delivered orders: {len(delivered):,}")

# DECISION 2: From delivered orders, drop rows where delivery date
# is missing — these are data errors (status says delivered but
# no delivery date recorded)
before = len(delivered)
delivered = delivered.dropna(
    subset=['order_delivered_customer_date',
            'order_estimated_delivery_date']
)
print(f"After dropping missing delivery dates: {len(delivered):,}")
print(f"Rows dropped: {before - len(delivered):,}")

# DECISION 3: Drop rows where delivery happened BEFORE purchase
# These are impossible and are data entry errors
impossible = (
    delivered['order_delivered_customer_date']
    < delivered['order_purchase_timestamp']
)
print(f"\nImpossible deliveries to remove: {impossible.sum()}")
delivered = delivered[~impossible]
print(f"Final clean delivered orders: {len(delivered):,}")

All orders:       99,440
Delivered orders: 96,477
After dropping missing delivery dates: 96,469
Rows dropped: 8

Impossible deliveries to remove: 0
Final clean delivered orders: 96,469


Calculate new columns using numpy — the feature engineering step

Turn raw dates into numbers your analysis can actually use

In [16]:
# COLUMN 1: How many days did delivery actually take?
delivered['actual_delivery_days'] = (
    delivered['order_delivered_customer_date']
    - delivered['order_purchase_timestamp']
).dt.days
# .dt.days converts a timedelta (time difference) to an integer

# COLUMN 2: Was the delivery late? (True/False)
delivered['is_late'] = (
    delivered['order_delivered_customer_date']
    > delivered['order_estimated_delivery_date']
)

# COLUMN 3: By how many days was it late (or early)?
# Positive = late. Negative = arrived early.
delivered['delivery_delay_days'] = (
    delivered['order_delivered_customer_date']
    - delivered['order_estimated_delivery_date']
).dt.days

# COLUMN 4: What time of day was the order placed?
delivered['order_hour'] = delivered[
    'order_purchase_timestamp'].dt.hour

# COLUMN 5: What day of the week?
delivered['order_dayofweek'] = delivered[
    'order_purchase_timestamp'].dt.day_name()

# COLUMN 6: Month and year (for trend charts)
delivered['order_month'] = delivered[
    'order_purchase_timestamp'].dt.to_period('M')

# --- numpy is faster than pandas for conditional columns ---
# COLUMN 7: Label late vs on time using numpy.where
# Format: np.where(condition, value_if_true, value_if_false)
delivered['delivery_status'] = np.where(
    delivered['is_late'], 'Late', 'On Time'
)

# COLUMN 8: Bucket delivery delay into readable groups
# np.select lets you handle multiple conditions at once
conditions = [
    delivered['delivery_delay_days'] < -7,
    delivered['delivery_delay_days'].between(-7, 0),
    delivered['delivery_delay_days'].between(1, 7),
    delivered['delivery_delay_days'] > 7
]
labels = ['Early (7+ days)', 'On time or early (<7d)',
          'Slightly late (1-7d)', 'Very late (7+ days)']

delivered['delay_bucket'] = np.select(conditions, labels,
                                       default='Unknown')

# Check your new columns
print(delivered[['actual_delivery_days','is_late',
                 'delivery_delay_days','delivery_status',
                 'delay_bucket']].head(10))
print(f"\nLate delivery rate: {delivered['is_late'].mean():.1%}")


   actual_delivery_days  is_late  delivery_delay_days delivery_status  \
0                     6    False                  -14         On Time   
1                     6    False                  -19         On Time   
2                     3    False                   -6         On Time   
3                    13    False                   -9         On Time   
4                    22    False                  -11         On Time   
5                    12    False                   -6         On Time   
6                     7    False                   -6         On Time   
7                     7    False                  -15         On Time   
8                    20    False                   -5         On Time   
9                     9    False                  -14         On Time   

             delay_bucket  
0         Early (7+ days)  
1         Early (7+ days)  
2  On time or early (<7d)  
3         Early (7+ days)  
4         Early (7+ days)  
5  On time or early (<7d)  


np.where and np.select are the numpy tools you use most in data cleaning. They are 10–50x faster than using df.apply() with a lambda for the same job on large datasets.


Clean and aggregate the payments table

One order can have multiple payment rows — you must collapse them first

In [17]:
# PROBLEM: payments has one row per installment
# An order with 6 installments has 6 rows
# You cannot just join this directly — you need order-level totals first

# Step 1: Check how many orders have multiple payment rows
payment_counts = payments.groupby('order_id').size()
print("Orders with 1 payment row: ",
      (payment_counts == 1).sum())
print("Orders with 2+ payment rows:",
      (payment_counts > 1).sum())

# Step 2: Collapse to one row per order
payment_totals = (
    payments
    .groupby('order_id')
    .agg(
        total_payment   = ('payment_value',   'sum'),
        num_installments= ('payment_installments', 'max'),
        payment_type    = ('payment_type',    'first')
    )
    .reset_index()
)

print(f"\nPayment totals shape: {payment_totals.shape}")
print(payment_totals.head())

# Step 3: Handle nulls in payment_value
print(f"\nNull payment values: {payments['payment_value'].isnull().sum()}")
# If any, fill with 0 — a null payment value means 0 was recorded
payment_totals['total_payment'] = (
    payment_totals['total_payment'].fillna(0)
)

Orders with 1 payment row:  96478
Orders with 2+ payment rows: 2961

Payment totals shape: (99439, 4)
                           order_id  total_payment  num_installments  \
0  00010242fe8c5a6d1ba2dd792cb16214      72.190002                 2   
1  00018f77f2f0320c557190d7a144bdd3     259.829987                 3   
2  000229ec398224ef6ca0657da4fc703e     216.869995                 5   
3  00024acbcdf0a6daa1e931b038114c75      25.780001                 2   
4  00042b26cf59d7ce69dfabb4e55b4fd9     218.039993                 3   

  payment_type  
0  credit_card  
1  credit_card  
2  credit_card  
3  credit_card  
4  credit_card  

Null payment values: 0



Clean the reviews table and handle missing comments

review_score nulls and missing text comments need different treatment



In [18]:
# Check what is missing
print("Missing review scores:  ",
      reviews['review_score'].isnull().sum())
print("Missing review comments:",
      reviews['review_comment_message'].isnull().sum())

# DECISION: rows with no review_score are useless for rating analysis
# Drop them
reviews_clean = reviews.dropna(subset=['review_score']).copy()
print(f"\nReviews after dropping null scores: {len(reviews_clean):,}")

# DECISION: missing comments just mean the customer didn't write one
# Fill with empty string — do NOT drop these rows
reviews_clean['review_comment_message'] = (
    reviews_clean['review_comment_message'].fillna('')
)

# Validate review scores are all between 1 and 5
invalid = ~reviews_clean['review_score'].between(1, 5)
print(f"Invalid scores (outside 1-5): {invalid.sum()}")

# One review per order — keep the most recent if duplicates exist
reviews_clean = (
    reviews_clean
    .sort_values('order_id')
    .drop_duplicates(subset='order_id', keep='last')
)
print(f"Reviews after deduplication: {len(reviews_clean):,}")
7

Missing review scores:   0
Missing review comments: 58255

Reviews after dropping null scores: 99,223
Invalid scores (outside 1-5): 0
Reviews after deduplication: 98,672


7

Merge everything into one master DataFrame

This is the single file you will use for all charts, Power BI, and ML



In [20]:
# ── FIXED Step 7 — merge everything into master DataFrame ──────────

# Step A: Payment totals — one row per order (you already have this)
payment_totals = (
    payments
    .groupby('order_id')['payment_value']
    .sum()
    .reset_index()
    .rename(columns={'payment_value': 'total_payment'})
)

# Convert payment_value to numeric first — it may still be a string
# because we reloaded with header=None
payment_totals['total_payment'] = pd.to_numeric(
    payment_totals['total_payment'], errors='coerce'
)

# Step B: Review scores — one row per order
review_scores = (
    reviews
    .dropna(subset=['review_score'])
    .groupby('order_id')['review_score']
    .mean()
    .reset_index()
)

# Convert review_score to numeric too
review_scores['review_score'] = pd.to_numeric(
    review_scores['review_score'], errors='coerce'
)

# Step C: Category per order — FIXED lambda
# The problem: mode()[0] crashes when all values are NaN
# The fix: wrap in a try/except so it safely returns 'unknown' instead
def safe_mode(x):
    try:
        m = x.dropna().mode()   # drop NaNs before mode
        if len(m) > 0:
            return m.iloc[0]    # iloc[0] is safer than [0]
        return 'unknown'
    except Exception:
        return 'unknown'

items_category = (
    items
    .groupby('order_id')['category']
    .agg(safe_mode)
    .reset_index()
)

# Step D: Merge everything onto delivered
master = (
    delivered
    .merge(payment_totals,  on='order_id', how='left')
    .merge(review_scores,   on='order_id', how='left')
    .merge(items_category,  on='order_id', how='left')
)

# Step E: Fill remaining nulls sensibly
master['category']      = master['category'].fillna('unknown')
master['review_score']  = pd.to_numeric(master['review_score'],
                                         errors='coerce')
master['total_payment'] = pd.to_numeric(master['total_payment'],
                                         errors='coerce')

# Step F: Verify
print(f"Final master shape: {master.shape}")
print(f"\nNull counts:")
print(master.isnull().sum())
print(f"\nFirst 3 rows of key columns:")
print(master[['order_id', 'total_payment',
              'review_score', 'category',
              'delivery_status']].head(3))

Final master shape: (96469, 20)

Null counts:
order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 14
order_delivered_customer_date      0
order_estimated_delivery_date      0
customer_state                     0
customer_city                      0
actual_delivery_days               0
is_late                            0
delivery_delay_days                0
order_hour                         0
order_dayofweek                    0
order_month                        0
delivery_status                    0
delay_bucket                       0
total_payment                      2
review_score                     647
category                           0
dtype: int64

First 3 rows of key columns:
                           order_id  total_payment  review_score  \
0  d2b0ee54c610c7e21c4ba15a4e5aafe2     163.289993           5.0   
1  694e3d121f16bcbb9b4595b6c6ab3baf


Final quality check and save

Validate your master table, then save it — this CSV feeds everything else

In [21]:
# Run this full validation before saving
print("="*50)
print("MASTER DATAFRAME VALIDATION")
print("="*50)
print(f"\nTotal rows:     {len(master):,}")
print(f"Total columns:  {master.shape[1]}")

print(f"\nDate range: "
      f"{master['order_purchase_timestamp'].min().date()} to "
      f"{master['order_purchase_timestamp'].max().date()}")

print(f"\nLate delivery rate:     {master['is_late'].mean():.1%}")
print(f"Avg delivery days:      {master['actual_delivery_days'].mean():.1f}")
print(f"Avg review score:       {master['review_score'].mean():.2f}")
print(f"Avg order value (R$):   {master['total_payment'].mean():.2f}")
print(f"Median order value (R$):{master['total_payment'].median():.2f}")

print(f"\nDelivery status counts:")
print(master['delivery_status'].value_counts())

print(f"\nTop 5 product categories:")
print(master['category'].value_counts().head())

print(f"\nRemaining nulls:")
nulls = master.isnull().sum()
print(nulls[nulls > 0])

# Save to CSV
master.to_csv('olist_master_clean.csv', index=False)
print(f"\nSaved: olist_master_clean.csv")

MASTER DATAFRAME VALIDATION

Total rows:     96,469
Total columns:  20

Date range: 2016-09-15 to 2018-08-29

Late delivery rate:     8.1%
Avg delivery days:      12.1
Avg review score:       4.16
Avg order value (R$):   159.85
Median order value (R$):105.28

Delivery status counts:
delivery_status
On Time    88643
Late        7826
Name: count, dtype: int64

Top 5 product categories:
category
bed_bath_table           9240
health_beauty            8621
sports_leisure           7476
computers_accessories    6519
furniture_decor          6208
Name: count, dtype: int64

Remaining nulls:
order_approved_at     14
total_payment          2
review_score         647
dtype: int64

Saved: olist_master_clean.csv
